In [0]:

raw_path = "/Volumes/de_dev/bronze/olist_raw_files/olist_order_reviews_dataset.csv"

with open(raw_path, "r", encoding="utf-8-sig", newline="") as f:
    for line_number, line in enumerate(f):
        if line_number < 8:
            print(f"Line {line_number + 1}: {line.rstrip()}")


In [0]:

import csv

raw_path = "/Volumes/de_dev/bronze/olist_raw_files/olist_order_reviews_dataset.csv"

target_ids = {
    "1fb4ddc969e6bea80e38deec00393a6f",
    "dbdf1ea31790c8ecfcc6750525661a9b",
    "ddc52555ca27b0fe67d5255147682d2d",
}

with open(raw_path, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    print("CSV columns:", reader.fieldnames)

    for row_number, row in enumerate(reader, start=2):
        if row.get("review_id") in target_ids:
            print(f"Row {row_number}: {row}")


In [0]:

from pyspark.sql import functions as F

raw_path = "/Volumes/de_dev/bronze/olist_raw_files/olist_order_reviews_dataset.csv"

review_test = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .load(raw_path)
)

print("Columns:", review_test.columns)
print("Parsed row count:", review_test.count())

review_test.select(
    "review_id",
    "order_id",
    "review_score",
    "review_creation_date",
    "review_answer_timestamp"
).filter(
    F.col("review_id").isNull()
    | ~F.col("review_id").rlike("^[0-9a-f]{32}$")
    | ~F.col("order_id").rlike("^[0-9a-f]{32}$")
    | ~F.col("review_score").rlike("^[1-5]$")
).show(20, truncate=False)
